In [1]:
import ray
import socket
import torch
import torch.nn as nn
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer
import ray.train.torch

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        ...
    def forward(self, x):
        ...

# 1. Define the distributed training loop
def train_func(config):
    # Fetch the specific GPU assigned to this worker by Ray
    device = ray.train.torch.get_device()
    
    # Get the hostname so we can verify which node this worker is on
    node_name = socket.gethostname()
    print(f"🚀 Worker booted on Node: {node_name} | Assigned Device: {device}")

    # Set up a basic PyTorch model
    model = nn.Linear(1, 1)
    
    # [CRITICAL STEP]: Prepare the model for Distributed Data Parallel (DDP)
    # This wraps the model so gradients synchronize across all 4 GPUs / 2 nodes.
    model = ray.train.torch.prepare_model(model)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    # Create dummy data and explicitly move it to the assigned GPU device
    X = torch.tensor([[1.0], [2.0], [3.0]]).to(device)
    y = torch.tensor([[2.0], [4.0], [6.0]]).to(device)

    epochs = config.get("num_epochs", 10)

    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X)
        loss = loss_fn(predictions, y)
        loss.backward()
        optimizer.step()

        # Report metrics back to the head node
        ray.train.report({
            "loss": loss.item(), 
            "epoch": epoch,
            "node": node_name
        })

# 2. Connect to your already-running Ray cluster
if not ray.is_initialized():
    # 'auto' tells Ray to connect to the existing daemon on this head node
    ray.init(address="auto") 

# 3. Configure scaling for 4 GPUs total
# Ray's scheduler will automatically see you need 4 GPUs. 
# It will place 2 workers on the head node's GPUs and 2 on the worker node's GPUs.
scaling_config = ScalingConfig(
    num_workers=4, 
    use_gpu=True
)

# 4. Initialize the Trainer
trainer = TorchTrainer(
    train_loop_per_worker=train_func,
    train_loop_config={"num_epochs": 15},
    scaling_config=scaling_config
)

# 5. Execute the distributed training
print("Starting distributed multi-GPU training...")
result = trainer.fit()

# 6. View the results
print("\nTraining completed successfully!")
print(f"Final reported loss: {result.metrics['loss']:.4f}")

2026-05-20 21:13:05,567	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-05-20 21:13:05,597	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 
/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Starting distributed multi-GPU training...


(TrainController pid=236423) Requesting resources: {'GPU': 1} * 4
(TrainController pid=236423) Attempting to start training worker group of size 4 with the following resources: [{'GPU': 1}] * 4
(RayTrainWorker pid=236571) Setting up process group for: env:// [rank=0, world_size=4]
(RayTrainWorker pid=64492, ip=10.0.1.34) 🚀 Worker booted on Node: digilab-transmit | Assigned Device: cuda:0
(RayTrainWorker pid=64492, ip=10.0.1.34) Moving model to device: cuda:0
(RayTrainWorker pid=64492, ip=10.0.1.34) Wrapping provided model in DistributedDataParallel.
(TrainController pid=236423) Started training worker group of size 4: 
(TrainController pid=236423) - (ip=10.0.1.14, pid=236571) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=236423) - (ip=10.0.1.14, pid=236572) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=236423) - (ip=10.0.1.34, pid=64492) world_rank=2, local_rank=0, node_rank=1
(TrainController pid=236423) - (ip=10.0.1.34, pid=64491) world_rank=3, local_ran


Training completed successfully!


TypeError: 'NoneType' object is not subscriptable